#### Tokenizer

In [2]:
from pathlib import Path
from tokenizers import ByteLevelBPETokenizer, Tokenizer
from tokenizers.processors import TemplateProcessing
from tqdm import tqdm


class BPETokenizer:
    def __init__(self, save_dir: str, vocab_size: int = 32000, min_frequency: int = 2):
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.tokenizer_path = self.save_dir / "tokenizer.json"
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        

    def train(self, dataset):
        tokenizer = ByteLevelBPETokenizer()

        def text_iterator():
            for example in tqdm(dataset, total=len(dataset), desc="Training BPE"):
                yield example["text"]

        tokenizer.train_from_iterator(
            text_iterator(),
            vocab_size=self.vocab_size,
            min_frequency=self.min_frequency,
            special_tokens=["<PAD>", "<BOS>", "<EOS>"],
        )

        # Add post-processor  
        tokenizer.post_processor = TemplateProcessing(
            single="<BOS> $A <EOS>",
            special_tokens=[
                ("<BOS>", tokenizer.token_to_id("<BOS>")),
                ("<EOS>", tokenizer.token_to_id("<EOS>")),
            ],
        )

        tokenizer.save(str(self.tokenizer_path))
        print("Tokenizer saved!")

        return tokenizer

    def from_file(self):
        if not self.tokenizer_path.exists():
            raise FileNotFoundError(f"{self.tokenizer_path} not found")

        tokenizer = Tokenizer.from_file(str(self.tokenizer_path))
        return tokenizer

#### Training tokenizer

In [15]:
tokenizer = BPETokenizer(save_dir='./tokenizer/bpe_tokenizer_v3', vocab_size=32000, min_frequency=2)
# bpe_tokenizer = tokenizer.train(combined_train_ds['train']) # (clean_ds)

Download tokenizer from file

In [ ]:
bpe_tokenizer = tokenizer.from_file()

print(bpe_tokenizer.get_vocab_size())
encoded = bpe_tokenizer.encode("Привет мир")
print(encoded.ids)
print(encoded.tokens)

#### class Dataset for big dataset

In [5]:
import os
import torch
import numpy as np

class LMDataset(torch.utils.data.Dataset):
    """ 
    Dataset only provides token sequences.
    The autoregressive shift is applied inside the model during loss computation, 
    keeping data loading independent from training objective.
    """
    def __init__(self, path, block_size=512, dtype=np.uint16):
        self.block_size = block_size
        file_size = os.path.getsize(path)
        self.total_tokens = file_size // np.dtype(dtype).itemsize
        self.data = np.memmap(path, dtype=dtype, mode="r")

    def __len__(self):
        return self.total_tokens // self.block_size

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size

        chunk = torch.from_numpy(self.data[start:end].astype(np.int64))

        return {"input_ids": chunk}

In [6]:
train_ds = LMDataset(
    path='./data/embed_corpus_train',
    block_size=1024,
    dtype=np.uint16
)

valid_ds = LMDataset(
    path='./data/embed_corpus_valid',
    block_size=1024,
    dtype=np.uint16
)
valid_subset = torch.utils.data.Subset(valid_ds, range(200))

test_ds = LMDataset(
    path='./data/embed_corpus_test',
    block_size=1024,
    dtype=np.uint16
)

print(len(train_ds))
print(train_ds[0]["input_ids"].shape)

886894
torch.Size([1024])


In [7]:
from torch.utils.data import DataLoader
BATCH_SIZE = 4

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

valid_loader = DataLoader(
    valid_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)
print(len(train_loader))
print(len(valid_loader))
print(len(test_loader))

221724
50
4475


#### class Dataset for small dataset

In [10]:
import torch
from torch.utils.data import Dataset
import numpy as np


class SMDataset(Dataset):
    def __init__(self, ds, tokenizer, block_size=512):
        self.block_size = block_size

        tokens = []

        # tokenize each article
        for row in ds:
            ids = tokenizer.encode(row['text']).ids
            tokens.extend(ids)

        self.tokens = np.array(tokens, dtype=np.int64)
        self.n_chunks = len(self.tokens) // block_size

    def __len__(self):
        return self.n_chunks

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size

        chunk = torch.from_numpy(self.tokens[start:end])

        return {"input_ids": chunk}

In [ ]:
gen_dataset = SMDataset(
    sap_ds,
    bpe_tokenizer,
    block_size=512
)
valid_dataset = SMDataset(
    valid_ds,
    bpe_tokenizer,
    block_size=512
)
print(len(gen_dataset))
print(len(valid_dataset))
print(gen_dataset[0]["input_ids"].shape)

In [8]:
batch = next(iter(train_loader))

print(batch["input_ids"].shape)
batch["input_ids"][0] 

torch.Size([4, 1024])


tensor([  912,   396, 13270,  ...,  8760,   367, 19036])

In [1]:
from tokenizer import BPETokenizer
bpe_tokenizer = BPETokenizer().from_file("tokenizer/bpe_tokenizer_v3/tokenizer.json")
bpe_tokenizer.post_processor = None

print(bpe_tokenizer.get_vocab_size())
encoded = bpe_tokenizer.encode("Привет мир")
print(encoded.ids)
print(encoded.tokens)

32000
[22984, 309, 2245]
['ÐŁÑĢÐ¸Ð²', 'ÐµÑĤ', 'ĠÐ¼Ð¸ÑĢ']


#### Rotary Position Embedding (RoPE)

In [1]:
import torch
import torch.nn as nn

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()
        self.dim = dim
        self.base = base

        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)

        self._seq_len = None
        self._cos = None
        self._sin = None

    def forward(self, seq_len, dtype):
        device = self.inv_freq.device
        if (
            seq_len != self._seq_len
            or self._cos.device != device
            or self._cos.dtype != dtype
        ):
            positions = torch.arange(seq_len, device=device).float()
            theta = torch.outer(positions, self.inv_freq)
            self._cos = theta.cos().to(dtype).unsqueeze(0).unsqueeze(0)
            self._sin = theta.sin().to(dtype).unsqueeze(0).unsqueeze(0)
            self._seq_len = seq_len

        return self._cos, self._sin
    

def apply_rope(q, k, cos, sin):
    # q, k: (B, H, T, D)
    # cos, sin: (1, 1, T, D/2)

    # Чётные и нечётные координаты (q1, q2) -> (x0​,x1​), (x2​,x3​), ...
    q1 = q[..., ::2]   # q1 = (B, H, T, D/2),  D/2 -> [ x0, x2, x4, ... ] 
    q2 = q[..., 1::2]  # q1 = (B, H, T, D/2),  D/2 -> [ x1, x3, x5, ... ]
    k1 = k[..., ::2]   # k1 = (B, H, T, D/2),  D/2 -> [ x0, x2, x4, ... ] 
    k2 = k[..., 1::2]  # k1 = (B, H, T, D/2),  D/2 -> [ x1, x3, x5, ... ]

    q_rot = torch.empty_like(q)
    k_rot = torch.empty_like(k)

    # Rotation q and k, broadcast (1,1,T,D/2)→(B,H,T,D/2)
    q_rot[..., ::2] = q1 * cos - q2 * sin
    q_rot[..., 1::2] = q1 * sin + q2 * cos

    k_rot[..., ::2] = k1 * cos - k2 * sin
    k_rot[..., 1::2] = k1 * sin + k2 * cos

    return q_rot, k_rot

#### Multi Head Self Attention

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout_rate=0.1):
        super().__init__()
        self.n_heads = n_heads
        
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.head_dim = d_model // n_heads

        # Positional embedding RoPE
        self.rope = RotaryEmbedding(self.head_dim)

        # Attantion layer
        self.qkv = nn.Linear(d_model, 3 * d_model)
        # Output layer
        self.out = nn.Linear(d_model, d_model)
        # Dropout layer 
        self.attn_drop = dropout_rate  # nn.Dropout(p=dropout_rate)
        self.dropout = nn.Dropout(p=dropout_rate)

        # self.register_buffer(
        #     "mask", 
        #     torch.tril(torch.ones(max_seq_len, max_seq_len)).bool()
        # )

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape
        qkv = self.qkv(x) # (batch, seq_len, 3*d_model)

        # Splits the last dimension into (n_heads, head_dim) for (q, k, v)
        qkv = qkv.view(batch_size, seq_len, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)  # (batch, seq_len, n_heads, head_dim)

        # Reshape for multi-head attention.
        q = q.transpose(1, 2)  # (batch, n_heads, seq_len, head_dim)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Apply RoPE 
        cos, sin = self.rope(seq_len, x.dtype)
        q, k = apply_rope(q, k, cos, sin)
        
        # # Dot-product attention logits  (batch, n_heads, seq_len, seq_len)
        # logits = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # # Apply causal (look-ahead) mask.  
        # mask = self.mask[:seq_len, :seq_len]
        # logits_masked = logits.masked_fill(~mask, -torch.inf) 

        # # Apply softmax to get attention weights.
        # attention_weights = torch.softmax(logits_masked, dim=-1)

        # # Apply attention dropout
        # attention_weights = self.attn_drop(attention_weights)

        # # Apply attention weights to values.
        # attn_out = attention_weights @ v  # (batch, n_heads, seq_len, head_dim)

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.attn_drop if self.training else 0.0,
            is_causal=True
        )

        # Concatenate heads and transpose back to (batch, seq_len, d_model)  
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        # output projection (mix heads)
        out = self.out(attn_out)
        
        # Apply dropout
        out = self.dropout(out)

        return out 

#### Transformer Block

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, max_seq_len)
        self.ln2 = nn.LayerNorm(d_model)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(0.1)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [4]:
class NanoGPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        max_seq_len,
    ):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, max_seq_len)
            for _ in range(n_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight # weight tying

    def forward(self, input_ids, labels=None):
        x = self.token_emb(input_ids)  # (batch, seq_len, d_model)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)  # (batch, seq_len, vocab_size)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()  # (batch, seq_len, vocab_size)
            shift_labels = labels[:, 1:].contiguous()      # (batch, seq_len)

            loss = nn.functional.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100
            )

        return logits, loss

In [ ]:
model = NanoGPT(bpe_tokenizer.get_vocab_size(), 384, 8, 6, 4*384, 1024)
res = model(batch["input_ids"], batch["input_ids"])
res

In [ ]:
embedding = nn.Embedding(bpe_tokenizer.get_vocab_size(), 384)
trn_block = TransformerBlock(384, 6, 4*384, 1024)
x = embedding(batch["input_ids"])
print(x.shape)
z = trn_block(x)
z

#### Model and optimizer initialization

In [15]:
import torch
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

run_name = "run_" + datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
writer = SummaryWriter(log_dir=f"./log/tensorboard/{run_name}")

NUM_EPOCHS = 2
GRAD_ACCUM_STEPS = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

model = NanoGPT(bpe_tokenizer.get_vocab_size(), 384, 8, 6, 4*384, 1024).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

total_steps = NUM_EPOCHS * len(train_loader)
warmup_steps = int(0.01 * total_steps)
decay_steps = total_steps - warmup_steps

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=1/4,
    end_factor=1.0,
    total_iters=warmup_steps
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=decay_steps,
    eta_min=1e-5
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)


#### Training loop

In [ ]:
model.train()

# checkpoint = torch.load("./checkpoints/best_checkpoint.pt")
# model.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
# scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
# start_epoch = checkpoint["epoch"] + 1
# if start_epoch > NUM_EPOCHS:
#     start_epoch = NUM_EPOCHS

log_step = 5 * GRAD_ACCUM_STEPS

best_loss = torch.inf
loss_list = []
optimizer.zero_grad(set_to_none=True)

for epoch in range(0, NUM_EPOCHS):
    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)
        logits, loss = model(input_ids, input_ids)

        loss_list.append(loss.item())

        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        # ---- optimizer step only every accum_steps ----
        if (step + 1) % GRAD_ACCUM_STEPS == 0:

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            if (step + 1) % log_step == 0:
                global_step = step + 1 + len(train_loader) * epoch
                lr = optimizer.param_groups[0]["lr"]
                avg_loss = np.mean(loss_list)
                print(f"epoch: {epoch}, step: {global_step} | loss: {avg_loss:.4f} | lr: {lr:.6e}")

                writer.add_scalar("train/loss", avg_loss, global_step)
                writer.add_scalar("train/lr", lr, global_step)
                loss_list = []

                # ---- validation ----
                model.eval()
                with torch.no_grad():
                    for step_v, batch_v in enumerate(valid_loader):
                        val_inp_ids = batch_v["input_ids"].to(device)
                        _, loss = model(val_inp_ids, val_inp_ids)
                        loss_list.append(loss.item())

                valid_avg_loss = np.mean(loss_list)
                print(f"epoch: {epoch}, step: {global_step} | valid_loss: {valid_avg_loss:.4f}")

                writer.add_scalar("valid/loss", valid_avg_loss, global_step)
                writer.flush()
                loss_list = []
                model.train()

                # ---- save best checkpoint ----
                if valid_avg_loss < best_loss:
                    best_loss = valid_avg_loss
                    checkpoint = {
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "epoch": epoch,
                    }
                    torch.save(checkpoint, "./utils/checkpoints/best_checkpoint.pt")            

writer.close()

In [ ]:
import torch
import numpy as np

checkpoint = torch.load("./checkpoints/best_checkpoint.pt")
model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
optim_step = checkpoint["optim_step"]

LOG_EVERY = 10    # optimizer steps
CLIP = 1.0

best_loss = float("inf")
train_losses = []
# optim_step = 0

optimizer.zero_grad(set_to_none=True)

def run_validation():
    model.eval()
    val_losses = []

    with torch.no_grad():
        for i, batch in enumerate(valid_loader):
            inp = batch["input_ids"].to(device)
            _, loss = model(inp, inp)
            val_losses.append(loss.item())

    model.train()
    return np.mean(val_losses)


for epoch in range(NUM_EPOCHS):

    for step, batch in enumerate(train_loader):

        input_ids = batch["input_ids"].to(device)

        logits, loss = model(input_ids, input_ids)

        train_losses.append(loss.item())

        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        # ---- optimizer step ----
        if (step + 1) % GRAD_ACCUM_STEPS == 0:

            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            optim_step += 1

            # ---- logging ----
            if optim_step % LOG_EVERY == 0:

                avg_train_loss = np.mean(train_losses)
                train_losses = []

                lr = optimizer.param_groups[0]["lr"]

                print(
                    f"epoch {epoch} | step {optim_step} | "
                    f"train_loss {avg_train_loss:.4f} | lr {lr:.2e}"
                )

                writer.add_scalar("train/loss", avg_train_loss, optim_step)
                writer.add_scalar("train/lr", lr, optim_step)

                # ---- validation ----
                val_loss = run_validation()

                print(
                    f"epoch {epoch} | step {optim_step} | "
                    f"valid_loss {val_loss:.4f}"
                )

                writer.add_scalar("valid/loss", val_loss, optim_step)
                writer.flush()

                # ---- best checkpoint ----
                if val_loss < best_loss:

                    best_loss = val_loss

                    checkpoint = {
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "scheduler_state_dict": scheduler.state_dict(),
                        "epoch": epoch,
                        "optim_step": optim_step,
                    }

                    torch.save(checkpoint, "./checkpoints/best_checkpoint.pt")

In [95]:
@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 10,
    temperature=0.8, 
    top_k=40,
    device: str = "cpu"
):
    model.eval()
    new_tokens = []

    input_ids = torch.tensor(
        tokenizer.encode(prompt).ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)  # (1, seq_len)

    for _ in range(max_new_tokens):

        logits, _ = model(input_ids)

        # берём последний токен
        next_token_logits = logits[:, -1, :]  # (1, vocab_size)

        # greedy
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        new_tokens.append(next_token)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(new_tokens)

In [31]:
prompt1= """<BOS>
<C>
<D>Компания выпускает облигации для привлечения капитала.</D>
</C>

<Q>
Зачем компания выпускает облигации?
</Q>

<A>

"""
#################################################
prompt2= """<BOS>
<C>
<D>В системе SAP различают несколько способов отражения затрат, и одним из ключевых является прямая проводка. Прямая проводка это фиксация суммы по счету с явным указанием номера счета, что означает, что пользователь или система явно выбирает конкретный основной счет для записи операции. Такой подход характерен прежде всего для первичных видов затрат, которые напрямую связаны с внешними операциями компании. Например, при начислении заработной платы или списании материалов бухгалтер указывает соответствующий счет, и сумма сразу отражается в учете. Это обеспечивает прозрачность и однозначность операций. Прямая проводка используется в модулях финансового учета и логистики и является базовым механизмом регистрации затрат.</D>
</C>

<Q>
Дай полный ответ только из контекста, не выдумывай: Какие есть способы отражения затрат в SAP?
</Q>

<A>

"""

##################################################
prompt3= """<BOS>
<C>
<D>Основное условное обозначение подшипника состоит из семи цифр основного условного обозначения (при нулевых значениях этих признаков оно может сокращаться до 2 знаков) и дополнительного обозначения, которое располагается слева и справа от основного. При этом дополнительное обозначение, расположенное слева от основного, всегда отделено знаком тире (—), а дополнительное обозначение, расположенное справа, всегда начинается с какой-либо буквы. Чтение знаков основного и дополнительного обозначения производится справа налево.</D>
<D>Транзакция KS02 используется для изменения мастер данных по МВЗ в SAP. С её помощью можно корректировать информацию о счетах затрат, назначениях и структурах контроля. Изменения возможны только пользователями с соответствующими правами.</D>
</C>

<Q>
Можно ли с помощью KS02 корректировать информацию о счетах затрат?
</Q>

<A>

"""

##################################################
prompt4= """<BOS>
<INST>Дай полный ответ, используя эти документы.</INST>

<CTX>
<D1>CO02 использует таблицы AUFK и AFKO для внесения изменений в существующие производственные заказы.</D>
<D1>Места возникновения затрат представляют собой организационные единицы, используемые для учета и контроля косвенных затрат в компании. Они создаются в рамках контроллинговой единицы и могут отражать различные аспекты структуры предприятия, такие как функциональные подразделения или производственные участки. Каждое МВЗ может быть связано с ответственным лицом и использоваться для планирования, учета и анализа затрат. Основной задачей МВЗ является обеспечение прозрачности распределения затрат внутри организации. При этом сами МВЗ не являются финансовыми счетами и не используются напрямую в финансовой отчетности, а служат инструментом управленческого учета.</D>
</CTX>

<Q>Какие таблицы используются в CO02?</Q>

<ANS>
"""

In [47]:
import json
with open("data/qa_datasets/qa_dataset.json", "r", encoding="utf-8") as f:
    qa_dataset = json.load(f)
sum(1 for x in qa_dataset if len(x["answer"]) > 60)

79223

In [51]:
import torch
import torch.nn as nn


def add_special_tokens_and_resize(
    model,
    tokenizer,
    special_tokens,
    init_std=0.001,
    verbose=True
):
    # ---- 1. Add tokens ----
    num_added = tokenizer.add_special_tokens(special_tokens)

    if verbose:
        print(f"Added {num_added} tokens")

    if num_added == 0:
        return model, tokenizer

    # ---- 2. Sizes ----
    old_vocab_size = model.token_emb.weight.shape[0]
    new_vocab_size = tokenizer.get_vocab_size()
    emb_dim = model.token_emb.weight.shape[1]

    if verbose:
        print(f"Old vocab: {old_vocab_size}, New vocab: {new_vocab_size}")

    # ---- 3. Resize embedding ----
    old_emb = model.token_emb
    new_emb = nn.Embedding(new_vocab_size, emb_dim)

    new_emb.weight.data[:old_vocab_size] = old_emb.weight.data
    nn.init.normal_(new_emb.weight.data[old_vocab_size:], mean=0.0, std=init_std)

    model.token_emb = new_emb

    # ---- 4. Resize lm_head ----
    old_head = model.lm_head

    new_head = nn.Linear(
        old_head.in_features,
        new_vocab_size,
        bias=False
    )

    new_head.weight.data[:old_vocab_size] = old_head.weight.data
    nn.init.normal_(new_head.weight.data[old_vocab_size:], mean=0.0, std=init_std)

    model.lm_head = new_head

    # ---- 5. Retie weights (IMPORTANT) ----
    model.lm_head.weight = model.token_emb.weight

    if verbose:
        print("Resize complete")

        for tok in special_tokens:
            print(tok, "->", tokenizer.encode(tok).ids)

    return model, tokenizer

In [52]:
import torch
from model import NanoGPT
from tokenizer import BPETokenizer

bpe_tokenizer = BPETokenizer().from_file("tokenizer/bpe_tokenizer_v1/tokenizer.json")

special_tokens = ["<INST>", "</INST>", "<CTX>", "</CTX>", 
                  "<D1>", "<D2>", "<D3>", "<D4>", "<D5>", "<D6>", "<D7>", "<D8>", "<D9>", "</D>", 
                  "<Q>", "</Q>", "<ANS>", "</ANS>", "<SRC>", "</SRC>"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NanoGPT(bpe_tokenizer.get_vocab_size(), 384, 8, 6, 4*384, 1024).to(device)

checkpoint = torch.load("utils/checkpoints/best_checkpoint_v3.pt")
model.load_state_dict(checkpoint["model_state_dict"])

model, tokenizer = add_special_tokens_and_resize(
    model,
    bpe_tokenizer,
    special_tokens
)

C:\Users\Sergey\AppData\Local\Temp\ipykernel_14140\831150609.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("utils/checkpoints/best_checkpoint_

Added 20 tokens
Old vocab: 32000, New vocab: 32020
Resize complete
<INST> -> [1, 32000, 2]
</INST> -> [1, 32001, 2]
<CTX> -> [1, 32002, 2]
</CTX> -> [1, 32003, 2]
<D1> -> [1, 32004, 2]
<D2> -> [1, 32005, 2]
<D3> -> [1, 32006, 2]
<D4> -> [1, 32007, 2]
<D5> -> [1, 32008, 2]
<D6> -> [1, 32009, 2]
<D7> -> [1, 32010, 2]
<D8> -> [1, 32011, 2]
<D9> -> [1, 32012, 2]
</D> -> [1, 32013, 2]
<Q> -> [1, 32014, 2]
</Q> -> [1, 32015, 2]
<ANS> -> [1, 32016, 2]
</ANS> -> [1, 32017, 2]
<SRC> -> [1, 32018, 2]
</SRC> -> [1, 32019, 2]


In [54]:
from training import build_optimizer, build_scheduler
optimizer = build_optimizer(model, lr=1e-4)
scheduler = build_scheduler(optimizer, 2000)

tokenizer.save("tokenizer/bpe_tokenizer_v2/tokenizer.json")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "optim_step": 0,
    "config": {
        "vocab_size": 32020,  # self.model.vocab_size,
        "d_model": 384,
        "n_layers": 8,
        "n_heads": 6,
        "d_ff": 4*384,
        "max_seq_len": 1024,
    },
}, "./utils/checkpoints/ckpt_pretrained_gpt.pt")

In [1]:
import torch
from model import NanoGPT
from tokenizer import BPETokenizer
bpe_tokenizer = BPETokenizer().from_file("tokenizer/bpe_tokenizer_v2/tokenizer.json")
bpe_tokenizer.post_processor = None
model = NanoGPT(bpe_tokenizer.get_vocab_size(), 384, 8, 6, 4*384, 1024).to('cuda')
# checkpoint = torch.load("utils/checkpoints/ckpt_pretrained_gpt.pt")
# model.load_state_dict(checkpoint["model_state_dict"])

In [2]:
import json
from torch.utils.data import DataLoader
from data import QADataset, qa_collate_fn

with open("data/qa_datasets/qa_dataset_val.json", "r", encoding="utf-8") as f:
    qa_dataset_val = json.load(f)

valid_ds = QADataset(
        data=qa_dataset_val,
        tokenizer=bpe_tokenizer,
        max_seq_len=1024
    )

pad_id = bpe_tokenizer.token_to_id("<PAD>")

valid_loader = DataLoader(
        valid_ds,
        batch_size=4,
        shuffle=True,
        collate_fn=lambda x: qa_collate_fn(x, pad_id)
    )

In [ ]:
from tqdm import tqdm
for sample in tqdm(valid_loader):
    qa = bpe_tokenizer.decode(sample["input_ids"].tolist()[0], skip_special_tokens=False).split('</Q>') 
    if '<ANS></ANS>' in qa[1] or '<ANS> </ANS>' in qa[1] or '<ANS>\n<SRC>' in qa[1]:
        print('empty answer')

In [48]:
sample = next(iter(valid_loader))
qa = bpe_tokenizer.decode(sample["input_ids"].tolist()[0], skip_special_tokens=False).split('<ANS>')  
prompt = qa[0]+'<ANS>'
prompt, qa[1]

('<BOS>\n<CTX>\n<D1>Основные данные МВЗ могут быть в активной и неактивной версии. Ведение основных данных МВЗ можно выполнять с помощью зависящих от времени зависимостей. В SAP-системе возможны следующие временные зависимости: не зависит от времени (I), зависимость от финансового года (Y), зависимость от периода (P), зависимость от дня (D). Основные данные, такие как имя, описание, ответственное лицо, отдел или вид места возникновения затрат, можно изменять любое число раз. Существуют ограничения на изменения, которые можно внести в область иерархии или валюту МВЗ.</D>\n<D2>Система работает только с активной версией основной записи МВЗ. Это означает, что для проводок используются только активные основные данные. Существуют активные и неактивные версии основных записей для МВЗ, бизнес-процессов и МВП. Невозможно создать неактивные основные данные с отдельной или групповой обработкой. Можно активировать или удалить соответствующие неактивные основные данные в организации предприятия, пр

In [30]:
prompt = """<BOS>
<INST>Дай полный ответ из контекста.</INST>

<CTX>
<D1>Места возникновения затрат представляют собой организационные единицы, используемые для учета и контроля косвенных затрат в компании. Они создаются в рамках контроллинговой единицы и могут отражать различные аспекты структуры предприятия, такие как функциональные подразделения или производственные участки. Каждое МВЗ может быть связано с ответственным лицом и использоваться для планирования, учета и анализа затрат. Основной задачей МВЗ является обеспечение прозрачности распределения затрат внутри организации. При этом сами МВЗ не являются финансовыми счетами и не используются напрямую в финансовой отчетности, а служат инструментом управленческого учета.</D>
<D2>Основые модули SAP:
- Учет по видам затрат (CO-OM-CEL)
Учет по видам затрат и выручки предоставляет обзор затрат и выручки, возникающих в организации.
- Учет по МВЗ (CO-OM-CCA)
Учет по местам возникновения затрат отслеживает затраты, возникающие в организации. Он полезен для присвоения косвенных затрат местоположению, в котором они возникли, в зависимости от источника.
- Внутренние заказы (CO-OM-OPA)
Внутренние заказы собирают затраты в соответствии с заданием, в котором они были выполнены. Этим заданиям можно присвоить бюджеты, чтобы гарантировать, что затраты не будут превышены.
- Контроллинг затрат на продукт (CO-PC)
Контроллинг затрат на продукт позволяет вычислить затраты, возникшие при производстве продукта или предоставлении услуги. Он позволяет рассчитать минимальную цену, по которой продукт может быть продан с прибылью.
- Учет результатов (CO-PA)
Учет результатов анализирует прибыли и убытки по отдельным сегментам рынка. Соответствующие затраты перерассчитываются на выручку для каждого сегмента рынка.
Учет результатов обеспечивает основу для принятия решений, например, для расчета цены, выбора клиента, определения условий и выбора каналов сбыта.
- Учет по МВП (EC-PCA)
В Учете по МВП анализируются прибыли и убытки независимых областей в организации, ответственных за их затраты и выручку.</D>
</CTX>

<Q>Перечисли все основные модули SAP?</Q>

<ANS> 
"""

prompt2 = """<BOS>
<INST>Дай полный ответ, используя эти документы.</INST>

<CTX>
<D1>Места возникновения затрат представляют собой организационные единицы, используемые для учета и контроля косвенных затрат в компании. Они создаются в рамках контроллинговой единицы и могут отражать различные аспекты структуры предприятия, такие как функциональные подразделения или производственные участки. Каждое МВЗ может быть связано с ответственным лицом и использоваться для планирования, учета и анализа затрат. Основной задачей МВЗ является обеспечение прозрачности распределения затрат внутри организации. При этом сами МВЗ не являются финансовыми счетами и не используются напрямую в финансовой отчетности, а служат инструментом управленческого учета.</D>
<D2>В системе SAP S/4HANA концепция видов затрат была существенно переработана по сравнению с предыдущими версиями. Теперь виды затрат больше не представлены как отдельные объекты, а интегрированы непосредственно в структуру основных счетов Главной книги. Это означает, что единый универсальный журнал содержит всю информацию как по финансовому учету, так и по контроллингу. Виды затрат фактически являются классификацией основных счетов, что позволяет использовать единый источник данных для анализа. Такая интеграция упрощает ведение данных, устраняет дублирование и повышает консистентность информации между модулями FI и CO. Основные счета теперь используются как для внешней отчетности, так и для внутреннего анализа затрат.</D>
<D3>Группы видов затрат используются исключительно для структурирования и удобства обработки данных. Они позволяют объединять различные виды затрат по определенным критериям, например по функциональному назначению или типу расходов. Такие группы применяются в отчетах, планировании и перерасчетах, позволяя обрабатывать сразу несколько видов затрат. Однако сами группы не являются видами затрат и не участвуют напрямую в проводках. Они служат вспомогательным инструментом, облегчающим работу пользователей и повышающим эффективность анализа.</D>
<D4>Тип вида затрат определяет, возможна ли прямая или косвенная проводка по основному счёту. Прямая проводка — фиксированная сумма проводится по счёту путём указания его номера — возможна для всех первичных видов затрат. Косвенная проводка — счёт определяется автоматически во время проводки — возможна только для вторичных видов затрат. Классификация основного счёта определяет операции, с которыми можно использовать вид затрат. Тип вида затрат можно просмотреть на вкладке Управляющие данные основной записи основного счёта.</D>
<D5>Основные счета с типом 01 — Первичные затраты и выручка, уменьшающая затраты — обозначают операционные расходы: заработную плату, расходы на продажи и административные затраты. Примеры: затраты на материал, на заработную плату, на энергию. Выручка, уменьшающая затраты, проводится с типом 01, а не 11, поскольку не является выручкой от продаж. Например, субаренда части офиса сокращает расходы на аренду, но не рассматривается как доход от продаж. Виды затрат типа 01 дебетуются для всех первичных проводок в FI или MM.</D>
</CTX>

<Q>Как теперь представлены виды затрат в системе SAP S/4HANA?</Q>

<ANS>
"""
prompt3 = """<BOS>
<CTX>
<D1>Группы видов затрат используются исключительно для структурирования и удобства обработки данных. Они позволяют объединять различные виды затрат по определенным критериям, например по функциональному назначению или типу расходов. Такие группы применяются в отчетах, планировании и перерасчетах, позволяя обрабатывать сразу несколько видов затрат. Однако сами группы не являются видами затрат и не участвуют напрямую в проводках. Они служат вспомогательным инструментом, облегчающим работу пользователей и повышающим эффективность анализа.</D>
<D2>Контроллинговая единица может содержать одну или несколько балансовых единиц, которые при необходимости могут работать в разных валютах. Все соответствующие балансовые единицы в контроллинговой единице должны использовать один и тот же оперативный план счетов.
Все операции внутреннего перерасчета относятся только к объектам одной контроллинговой единицы.
Внутренние бизнес-операции отображаются в контроллинговой единице. Первичные затраты переносятся из внешнего учета и отчетности и структурируются в соответствии с внутренними критериями. Если первичные затраты являются прямыми затратами, они присваиваются носителям затрат. Если они имеют носитель косвенных затрат, они присваиваются местам возникновения затрат или заказам на косвенные затраты, а затем перерассчитываются с помощью методов внутреннего перерасчета.
При создании основных данных система всегда присваивает объекты контроллинга контроллинговой единице и балансовой единице.
Контроллинговая единица — это организационная единица на предприятии, для которой можно выполнить полный учет затрат в закрытой системе.
Благодаря более детальному уровню детализации в Контроллинге (дополнительные структурные элементы, такие как места возникновения затрат или внутренние заказы), информация может быть получена из Контроллинга, например, для контроля затрат, контроллинга деятельности предприятия или контроллинга для сбыта.</D>
<D3>В системе SAP S/4HANA концепция видов затрат была существенно переработана по сравнению с предыдущими версиями. Теперь виды затрат больше не существуют как отдельные объекты, а интегрированы непосредственно в структуру основных счетов Главной книги. Это означает, что единый универсальный журнал содержит всю информацию как по финансовому учету, так и по контроллингу. Виды затрат фактически являются классификацией основных счетов, что позволяет использовать единый источник данных для анализа. Такая интеграция упрощает ведение данных, устраняет дублирование и повышает консистентность информации между модулями FI и CO. Основные счета теперь используются как для внешней отчетности, так и для внутреннего анализа затрат.</D>
</CTX>

<INST>Ответь детально, только из данных документов.</INST>

<Q> Поясни что такое контроллинговая единица?</Q>

<ANS>
"""
prompt4= """<BOS>
<C>
<D>Основное условное обозначение подшипника состоит из семи цифр основного условного обозначения (при нулевых значениях этих признаков оно может сокращаться до 2 знаков) и дополнительного обозначения, которое располагается слева и справа от основного. При этом дополнительное обозначение, расположенное слева от основного, всегда отделено знаком тире (—), а дополнительное обозначение, расположенное справа, всегда начинается с какой-либо буквы. Чтение знаков основного и дополнительного обозначения производится справа налево.</D>
<D>Транзакция KS02 используется для изменения мастер данных по МВЗ в SAP. С её помощью можно корректировать информацию о счетах затрат, назначениях и структурах контроля. Изменения возможны только пользователями с соответствующими правами.</D>
<D>На экране ME21N можно выбрать следующие поля: поставщик, организация закупок, группа закупок, дата и условия поставки для создания заказа на закупку.</D>
</C>

<Q>
Какие поля имеются на экране ME21N?
</Q>

<A>

"""
prompt5= """<BOS>
<C>
<D>Ароматизаторы (отдушки) — вещества, которые используют для придания продуктам или изделиям определённых запахов, создания или улучшения аромата. Ароматизаторами называют специальные изделия, предназначенные для придания определенного аромата воздуху в помещениях (напр. автомобильные ароматизаторы). Ароматизаторами также называют добавки, которые вводят в некоторые бытовые изделия (например, в изделия из пластмассы, резины для одорации или дезодорации, ароматизации синтетической кожи под натуральную и т. д.). Пищевые ароматизаторы добавляют к пищевым продуктам, кормам для животных, лекарственным средствам, средствам личной гигиены (например, зубной пасте) для придания им вкуса и запаха или для коррекции имеющегося вкуса и запаха.</D>
<D>Сходство химической структуры предопределяет одинаковый механизм действия всех -лактамов (нарушение синтеза клеточной стенки бактерий), а также перекрёстную аллергию к ним у некоторых пациентов. Пенициллины, цефалоспорины и монобактамы чувствительны к гидролизующему действию особых ферментов — -лактамаз, вырабатываемых рядом бактерий. Карбапенемы характеризуются значительно более высокой устойчивостью к -лактамазам. С учётом высокой клинической эффективности и низкой токсичности -лактамные антибиотики составляют основу антимикробной химиотерапии на современном этапе, занимая ведущее место при лечении большинства инфекций.</D>
<D>Бытовая химия (англ. household chemicals) — непродовольственные химические вещества, средства ухода за одеждой, помещениями, автомобилями, посудой и тому подобным, которые обычно встречаются и используются в домохозяйстве. К средствам бытовой химии также традиционно относят дезинфекторы, репелленты и другие химические средства, которые назначены, в частности, для очистки определенных поверхностей, борьбы с вредителями и общих гигиенических потребностей. ПФ краска, масляная и на водной основе. К бытовой химии не относятся товары и продукция парфюмерно-косметического назначения: парфюмерия, гигиеническая и декоративная косметика. По версии ЕЭС ПКТ подразделяют на парфюмерию; очищающие вещества; средства макияжа; средства защиты; освежающие средства. По классификатору ОКП, в основу которого положена общность технологии производства, выделяют одеколоны и душистые воды (9155), духи и эфирные масла в сувенирных наборах (9156), парфюмерные наборы и серии (9157), продукция косметическая (9158). Пищевые добавки, как правило, не подпадают под эту категорию, если только они не используются иначе, как для потребления человеком. Добавки в целом (например, стабилизаторы и красители, которые находятся в стиральных порошках и моечных средствах для посудомоечных машин) делают классификацию бытовой химии сложнее, к тому же некоторые из этих химикатов являются раздражителями или сильными аллергенами. Химические вещества бытовой химии, которые не компостируются, представляют серьезную экологическую опасность и опасность для здоровья человека. А в добавление к тому, что при проглатывании они имеют негативные токсичные эффекты (часто очень серьезные), химические вещества могут содержать легковоспламеняющиеся или коррозийные вещества.</D>
</C>

<Q>
К чему чувствительны пенициллины, цефалоспорины и монобактамы?
</Q>

<A>

"""
prompt6= """<BOS>
<CTX>
<D1>Европейский стандарт BS EN 50102:1995 "Степени защиты, предоставляемые защитами для электрооборудования от внешних механических воздействий (IK-код)", описывает стандарт присвоения IK-кода электрооборудованию, определяющего специфицированную степень закрывающей защиты для защиты содержимого от внешних ударов.</D>
<D2>CENELEC (фр. Comite Europeen de Normalisation Electrotechnique) — Европейский комитет электротехнической стандартизации, отвечающий за европейские стандарты в области электротехники. Вместе с ETSI (телекоммуникации) и CEN (другие технические области) CENELEC формирует европейскую систему технического нормирования и стандартизации. Стандарты этих учреждений согласуются регулярными принятиями стандартов во многих странах за пределами Европы, которые следуют европейским техническим стандартам. Хотя CENELEC работает в тесном сотрудничестве с Европейским союзом, он не является учреждением Европейского союза.</D>
<D3>Маркировка CE (аббревиатура фр. Conformite Europeenne — "европейское соответствие") — специальный знак, наносимый на изделие, который удостоверяет, что изделие соответствует основным требованиям директив ЕС и гармонизированным стандартам Европейского союза, а также то, что продукт прошёл процедуру оценки соответствия директивам. Маркировка CE указывает на то, что изделие не является вредным (опасным) для здоровья его потребителей, а также безвредно для окружающей среды. Однако следует учитывать, что знак CE не является символом качества продукции. Решение 768/2008/EC (DECISION No 768/2008/EC), принятое 9 июля 2008 года, регулирует права и обязанности по применению маркировки СЕ (CE Mark). Согласно данному Решению Европейского Парламента, существуют рекомендации странам Европейского союза по контролю над внутренним рынком ЕС по обороту продукции, подлежащей обязательной маркировке знаком СЕ. В странах Европейского Сообщества введены административные и уголовные наказания за нарушения правил, которые касаются применения маркировки СЕ. Продукция, не соответствующая директивам и гармонизированным стандартам Европейского союза, обязывающим нанесение знака СЕ (CE Mark), не допускается на внутренний рынок ЕС. Однако следует отметить, что не для всех групп товаров требуется нанесение маркировки СЕ. Знак СЕ является единственным знаком в странах Европейского союза, подтверждающим соответствие продукции европейским стандартам безопасности для человека, имущества и окружающей среды.</D>
</CTX>

<INST>Ответь детально, только из данных документов.</INST>

<Q>Кто отвечает за европейские стандарты в области электротехники?</Q>

<ANS>
"""

In [53]:
prompt = '<BOS>\n<CTX>\n<D1>Основные данные МВЗ могут быть в активной и неактивной версии. Ведение основных данных МВЗ можно выполнять с помощью зависящих от времени зависимостей. В SAP-системе возможны следующие временные зависимости: не зависит от времени (I), зависимость от финансового года (Y), зависимость от периода (P), зависимость от дня (D). Основные данные, такие как имя, описание, ответственное лицо, отдел или вид места возникновения затрат, можно изменять любое число раз. Существуют ограничения на изменения, которые можно внести в область иерархии или валюту МВЗ.</D>\n<D2>Система работает только с активной версией основной записи МВЗ. Это означает, что для проводок используются только активные основные данные. Существуют активные и неактивные версии основных записей для МВЗ, бизнес-процессов и МВП. Невозможно создать неактивные основные данные с отдельной или групповой обработкой. Можно активировать или удалить соответствующие неактивные основные данные в организации предприятия, при обработке стандартной иерархии для мест возникновения затрат, в стандартной иерархии для МВП, в обработке стандартной иерархии для бизнес-процессов, а также в пользовательской настройке соответствующего приложения.</D>\n<D3>Валюту объекта МВЗ можно изменить только в том случае, если плановые или фактические данные не проведены в МВЗ. Валюта объекта всегда действительна только для того финансового года, в котором она была создана, и изменить её в течение финансового года невозможно. Если активирован индикатор Другая валюта балансовой единицы, система автоматически копирует валюту балансовой единицы в качестве валюты объекта, и поле Валюта становится недоступным для ввода. Это ограничение действует для всех мест возникновения затрат в рамках контроллинговой единицы.</D>\n<D4>Присвоение места возникновения затрат узлу в стандартной иерархии или изменение валюты МВЗ возможно только в определённых случаях. Основные данные, такие как имя, описание, вид места возникновения затрат, отдел, ответственное лицо, индикатор или адрес, можно изменять любое число раз. Присвоение группы МВЗ определяется на протяжении всего жизненного цикла МВЗ и может быть изменено только в течение всего жизненного цикла. Если не выбрать весь жизненный цикл места возникновения затрат, система не примет изменение. При изменении присвоения группы система создаёт документ изменений.</D>\n<D5>Основная запись может иметь активную версию, неактивную версию или и то, и другое. При изменении одного или нескольких значений основных данных система сначала сохраняет их в новой неактивной версии, не выполняя необходимые проверки непротиворечивости. Эта версия не используется продуктивно, например для проводок. При создании новой основной записи система сначала создаёт неактивную версию. Для продуктивного использования основных данных необходимо активировать основную запись. При активации основных данных МВЗ система проверяет данные и, если противоречия не обнаружены, переносит их в активную версию основной записи. Если уже существует более старая активная версия соответствующей основной записи, она перезаписывается. Неактивные основные данные не переносятся и не распределяются через ALE.</D>\n<D6>Репрезентативные МВЗ присваиваются узлам стандартной иерархии или альтернативной иерархии. Невозможно выполнить проводку или планирование по репрезентативным МВЗ. SAP рекомендует объединить все репрезентативные МВЗ в один узел и присвоить его под верхним узлом в иерархии. Чтобы заблокировать репрезентативное МВЗ от проводок вручную, необходимо выбрать Индикаторы на основном экране МВЗ и установить индикаторы блокирования. Если во время ведения информации по отчёту вводится репрезентативное МВЗ, которое ещё не было создано как место возникновения затрат в контроллинговой единице, система выдаёт соответствующее сообщение.</D>\n<D7>В SAP S/4HANA универсальный журнал содержит поле Счёт, охватывающее как счета Главной книги, так и виды затрат. Виды затрат теперь являются частью плана счетов, поэтому больше не требуется выполнять ведение основных данных видов затрат по отдельности. Основные счета классифицируют оцененное потребление производственных факторов компании в контроллинговой единице и предоставляют информацию о стоимостном потоке и потреблении стоимости. Функции обработки видов затрат доступны через транзакцию FS00 или приложение SAP Fiori Управление планом счетов (F0763A). Виды затрат представлены в системе как вид основного счёта.</D>\n</CTX>\n\n<INST>Дай краткий ответ только из данных документов.</INST>\n\n<Q>Что происходит с неактивной версией основной записи МВЗ при активации основных данных в SAP S/4HANA?</Q>\n\n<ANS>'

In [55]:
params = sum(p.numel() for p in model.parameters())
params, f"{params / 1e6:.1f}M"

(26492160, '26.5M')

In [54]:
checkpoint = torch.load("./utils/checkpoints/best_checkpoint.pt", weights_only=True ) # map_location=torch.device('cpu'))
model.load_state_dict(checkpoint["model_state_dict"])

text = generate(
    model,
    bpe_tokenizer,
    prompt= prompt,
    max_new_tokens=150,
    temperature=0.01,
    top_p=1,
    repetition_penalty=1,
    device="cuda"   
)

print(text)

<BOS>
<CTX>
<D1>Основные данные МВЗ могут быть в активной и неактивной версии. Ведение основных данных МВЗ можно выполнять с помощью зависящих от времени зависимостей. В SAP-системе возможны следующие временные зависимости: не зависит от времени (I), зависимость от финансового года (Y), зависимость от периода (P), зависимость от дня (D). Основные данные, такие как имя, описание, ответственное лицо, отдел или вид места возникновения затрат, можно изменять любое число раз. Существуют ограничения на изменения, которые можно внести в область иерархии или валюту МВЗ.</D>
<D2>Система работает только с активной версией основной записи МВЗ. Это означает, что для проводок используются только активные основные данные. Существуют активные и неактивные версии основных записей для МВЗ, бизнес-процессов и МВП. Невозможно создать неактивные основные данные с отдельной или групповой обработкой. Можно активировать или удалить соответствующие неактивные основные данные в организации предприятия, при обр

In [5]:
import torch
import torch.nn.functional as F

def generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=50,
    temperature=0.5,
    top_p=0.9,
    repetition_penalty=1.0,
    device="cpu"
):
    model.eval()

    input_ids = torch.tensor(
        tokenizer.encode(prompt).ids,
        dtype=torch.long
    )[None].to(device)

    generated = input_ids.clone()
    end_tok_id = tokenizer.encode("<EOS>").ids[0]

    for _ in range(max_new_tokens):

        with torch.no_grad():
            logits, _ = model(generated)

        logits = logits[:, -1, :]

        if temperature == 0:
            next_token = torch.argmax(logits, dim=-1, keepdim=True)

        else:
            logits = logits / temperature

            if repetition_penalty != 1.0:
                for token_id in set(generated[0].tolist()):
                    if logits[0, token_id] > 0:
                        logits[0, token_id] /= repetition_penalty
                    else:
                        logits[0, token_id] *= repetition_penalty

            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            sorted_probs = F.softmax(sorted_logits, dim=-1)
            cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = False

            sorted_logits[sorted_indices_to_remove] = -float("inf")
            logits = torch.zeros_like(logits).scatter(1, sorted_indices, sorted_logits)

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1)

        generated = torch.cat([generated, next_token], dim=1)

        if generated[0, -1].item() == end_tok_id:
            break

    return tokenizer.decode(generated[0].tolist(), skip_special_tokens=False)

In [97]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=20, temperature=0.8, top_k=40, device="cpu"):

    model.eval()
    new_tokens = []

    input_ids = torch.tensor(
        tokenizer.encode(prompt).ids,
        dtype=torch.long
    )[None].to(device)

    for _ in range(max_new_tokens):

        logits, _ = model(input_ids)
        logits = logits[:, -1, :] / temperature

        if top_k:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, 1)
        new_tokens.append(next_token)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(new_tokens)

In [ ]:
# scaler = torch.amp.GradScaler()

# for batch in train_loader:
#     optimizer.zero_grad()

#     with torch.cuda.amp.autocast():  # матричные умножения → float16, softmax → float16, LayerNorm → float32
#         outputs = model(**batch)
#         loss = outputs.loss

#     scaler.scale(loss).backward()  # увеличивает значение на константу
#     scaler.step(optimizer)         # автоматически unscale + проверка overflow
#     scaler.update()                # адаптирует scale

# 1. Умножает loss на большой коэффициент (например 2¹⁶)
# 2. Делает backward
# 3. Делит градиенты обратно перед optimizer.step()
# 4. Проверяет NaN / inf
# 5. Автоматически регулирует scale